Train meta learner bằng tập train, early stop bằng valid sau đó test trên tập test.

In [1]:
import pandas as pd
data_valid = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet")
data_test = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet")
data_train = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet")

In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
import pandas as pd
import numpy as np
import copy
import gc
import os

# mô hình GatEmbedder
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = x.permute(0, 2, 1) 
        
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        

        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


# tạo đồ thị GAT cho tập valid và test (cần tách riêng để mô phỏng thực tế (không gộp lại giống lúc train))
def build_inference_graph(df, window_size=10):
    print("Pre-processing Timestamp cho Inference Graph...")
    if 'timestamp_num' not in df.columns:
        df['timestamp_num'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
        
    df.sort_values(by='timestamp_num', inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    cols_present = [c for c in cols_to_drop if c in df.columns]
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
    features_np = np.ascontiguousarray(features_np)
    
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()

    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    ip_to_indices = {}
    from tqdm.auto import tqdm
    for idx in tqdm(range(len(df)), desc="[1/2] Build IP Dictionary"):
        s_ip, d_ip = src_ips[idx], dst_ips[idx]
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx: ip_to_indices[s_ip].append(idx)
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx: ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Nối lưới đồ thị (1-chiều Inference)"):
        n_flows = len(indices)
        if n_flows < 2: continue
        for i in range(1, n_flows):
            curr_idx = indices[i]
            for j in range(max(0, i - window_size), i):
                past_idx = indices[j]
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx])
                if dt_raw > 60.0: continue
                dt = np.log1p(dt_raw * 1e6) / 18.0
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0
                attr = [dt, same_dir, same_sport, same_dport]
                
                all_src.append(past_idx)
                all_dst.append(curr_idx)
                all_edge_attrs.append(attr)
                
    df_edges = pd.DataFrame({'src': all_src, 'dst': all_dst, 'a0': [a[0] for a in all_edge_attrs], 'a1': [a[1] for a in all_edge_attrs], 'a2': [a[2] for a in all_edge_attrs], 'a3': [a[3] for a in all_edge_attrs]})
    df_edges.drop_duplicates(subset=['src', 'dst'], inplace=True)
    df_edges.reset_index(drop=True, inplace=True)
    
    edge_index = torch.tensor(df_edges[['src', 'dst']].values.T, dtype=torch.long).contiguous()
    edge_attr = torch.tensor(df_edges[['a0', 'a1', 'a2', 'a3']].values, dtype=torch.float32).contiguous()
    
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    graph.mask = torch.ones(len(df), dtype=torch.bool) 
    
    del features_np, df_edges, all_src, all_dst, all_edge_attrs
    gc.collect()
    return graph, df

# trích xuất embeddings GAT thực tế bằng NeighborLoader
@torch.no_grad()
def extract_embeddings_mask(model, graph_data, device='cpu'):
    model.eval()
    # Tăng mức lấy mẫu hàng xóm lên mức rất cao [100, 30] để bao quát được toàn bộ mạng lưới Botnet/DDoS lớn
    loader = NeighborLoader(graph_data, num_neighbors=[100, 30], input_nodes=graph_data.mask, batch_size=512, shuffle=False, num_workers=0)
    all_embeddings = []
    from tqdm.auto import tqdm
    pbar = tqdm(loader, desc="Extracting GAT Embeddings")
    for batch in pbar:
        batch = batch.to(device)
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        all_embeddings.append(emb[:batch.batch_size].cpu().numpy())
    pbar.close()
    return np.concatenate(all_embeddings, axis=0)

# trích xuất xác suất dự đoán từ mô hình CNN-BiLSTM-Attention
@torch.no_grad()
def extract_probs_cnn_bilstm(model, X_windows, batch_size=512, device='cpu'):
    model.eval()
    import numpy as np

    # Kiểm tra nhanh kích thước -> debug
    try:
        print("X_windows.shape:", X_windows.shape, "dtype:", X_windows.dtype, "bytes:", getattr(X_windows, "nbytes", None))
    except Exception:
        pass

    all_probs = []
    N = len(X_windows)
    for i in range(0, N, batch_size):
        batch_np = X_windows[i:i+batch_size]
        # đảm bảo float32 để giảm 2x bộ nhớ so với float64
        if batch_np.dtype != np.float32:
            batch_np = batch_np.astype(np.float32, copy=False)
        inputs = torch.from_numpy(batch_np).to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

In [ ]:
# GAT-only evaluation (node-level) — dán vào notebook
import os
import numpy as np
import torch
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch_geometric.loader import NeighborLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# ensure test dataframe exists
if 'data_test' not in globals():
    raise RuntimeError("Không tìm thấy `data_test` trong namespace. Chạy cell load dữ liệu trước (ví dụ đọc parquet).")

# build graph_test if missing
if 'graph_test' not in globals():
    print("Xây dựng `graph_test` từ `data_test` (hơi tốn RAM/time)...")
    graph, data_test = build_inference_graph(data_test)
else:
    graph = graph_test
    data_test = data_test

# ensure num_features/num_classes known
if 'num_features' not in globals():
    num_features = len([c for c in data_test.columns if c not in ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']])
if 'num_classes' not in globals():
    num_classes = len(data_test['label'].unique())

# load GAT model
gat_path = r"model_final\gat_embedder.pth"
if not os.path.exists(gat_path):
    raise FileNotFoundError(f"Không tìm thấy weights GAT tại {gat_path}")

gat_model = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=num_classes, heads=8, edge_dim=4).to(device)
gat_model.load_state_dict(torch.load(gat_path, map_location=device))
gat_model.eval()

@torch.no_grad()
def predict_gat_only(model, graph_data, device='cpu', batch_size=512):
    loader = NeighborLoader(graph_data, num_neighbors=[5, 3], input_nodes=graph_data.mask, batch_size=batch_size, shuffle=False, num_workers=0)
    preds = []
    for batch in tqdm(loader, desc="GAT predict"):
        batch = batch.to(device)
        out, _ = model(batch.x, batch.edge_index, batch.edge_attr)
        preds.append(out[:batch.batch_size].argmax(dim=1).cpu().numpy())
    return np.concatenate(preds, axis=0)

preds = predict_gat_only(gat_model, graph, device=device, batch_size=512)

# align ground-truth using graph.mask order
mask_array = graph.mask.cpu().numpy() if isinstance(graph.mask, torch.Tensor) else np.asarray(graph.mask)
mask_idx = np.where(mask_array)[0]
y_true = data_test['label'].values[mask_idx]

print("\nGAT-only evaluation (node-level)")
print(" - Samples evaluated:", len(y_true))
print("Accuracy:         ", f"{accuracy_score(y_true, preds)*100:.2f}%")
print("F1-Score (Macro): ", f"{f1_score(y_true, preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_true, preds, digits=4))

Device: cuda
Xây dựng `graph_test` từ `data_test` (hơi tốn RAM/time)...
Pre-processing Timestamp cho Inference Graph...


[1/2] Build IP Dictionary:   0%|          | 0/760240 [00:00<?, ?it/s]

[2/2] Nối lưới đồ thị (1-chiều Inference):   0%|          | 0/19 [00:00<?, ?it/s]

C:\Users\Admin\AppData\Local\Temp\ipykernel_19756\3856007947.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gat_model.load_state_dict(torch.load(gat_path, map_location

GAT predict:   0%|          | 0/1485 [00:00<?, ?it/s]


GAT-only evaluation (node-level)
 - Samples evaluated: 760240
Accuracy:          97.06%
F1-Score (Macro):  89.43%

Classification Report:

              precision    recall  f1-score   support

           0     0.7597    0.9769    0.8547     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.7815    1.0000    0.8774      2515
           3     0.9999    0.9792    0.9894     36253
           4     0.6607    0.8419    0.7404     18979
           5     0.9781    1.0000    0.9889      2451
           6     0.7573    0.7068    0.7311     11847
           7     1.0000    0.9999    1.0000    107436
           8     0.7845    0.9958    0.8776     16746
           9     0.9997    0.6667    0.7999     41523
          10     0.9994    0.9581    0.9783     18567

    accuracy                         0.9706    760240
   macro avg     0.8837    0.9205    0.8943    760240
weighted avg     0.9759    0.9706    0.9704    760240



In [ ]:
# GAT -> XGBoost evaluation (node-level)
import os
import numpy as np
import torch
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score, f1_score
import xgboost as xgb
from torch_geometric.loader import NeighborLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# load test dataframe if missing
if 'data_test' not in globals():
    import pandas as pd
    data_test = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet")

# build graph_test if missing
if 'graph_test' not in globals():
    print("Building graph_test from data_test...")
    graph, data_test = build_inference_graph(data_test)
else:
    graph = graph_test

# infer feature/label counts
num_features = len([c for c in data_test.columns if c not in ['label','flow_id','timestamp','timestamp_num','src_ip','dst_ip','protocol','src_port','dst_port']])
num_classes = int(data_test['label'].nunique())

# load GAT embedder
gat_path = r"model_final\gat_embedder.pth"
if not os.path.exists(gat_path):
    raise FileNotFoundError(f"Missing GAT weights: {gat_path}")
gat = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=num_classes, heads=8, edge_dim=4).to(device)
gat.load_state_dict(torch.load(gat_path, map_location=device))
gat.eval()

# extract GAT embeddings (uses helper if present)
if 'extract_embeddings_mask' in globals():
    test_gat_emb = extract_embeddings_mask(gat, graph, device=device)
else:
    # fallback extractor
    @torch.no_grad()
    def _extract(model, graph_data, device='cpu', batch_size=512):
        loader = NeighborLoader(graph_data, num_neighbors=[5,3], input_nodes=graph_data.mask, batch_size=batch_size, shuffle=False, num_workers=0)
        all_emb = []
        for batch in tqdm(loader, desc='Extract GAT emb'):
            batch = batch.to(device)
            _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
            all_emb.append(emb[:batch.batch_size].cpu().numpy())
        return np.concatenate(all_emb, axis=0)
    test_gat_emb = _extract(gat, graph, device=device)

print('GAT embeddings shape:', test_gat_emb.shape)

# load XGBoost trained on GAT embeddings
xgb_path = r"model_final\GAT_XGB_Hybrid_FAISS_Model.json"
if not os.path.exists(xgb_path):
    raise FileNotFoundError(f"Missing XGBoost model: {xgb_path}")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(xgb_path)

# predict and evaluate
y_pred = xgb_model.predict(test_gat_emb)

# align ground-truth using graph.mask
mask_array = graph.mask.cpu().numpy() if isinstance(graph.mask, torch.Tensor) else np.asarray(graph.mask)
mask_idx = np.where(mask_array)[0]
y_true = data_test['label'].values[mask_idx]

print("\nGAT + XGBoost evaluation (node-level)")
print("Samples:", len(y_true))
print("Accuracy:        ", f"{accuracy_score(y_true, y_pred)*100:.2f}%")
print("F1-Score (Macro):", f"{f1_score(y_true, y_pred, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
import numpy as np
from tqdm.auto import tqdm
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

def get_window_data_with_index(df, window_size=10):
    """
    trượt cửa sổ bằng chỉ số mảng (numpy array index) đẻ nối output của model (GAT_XGB)
    """
    ignore_cols = ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']
    feature_cols = [c for c in df.columns if c not in ignore_cols]
    
    features = df[feature_cols].values.astype(np.float32)
    labels = df['label'].values
    
    X_windows = []
    y_targets = []
    target_indices = []
    
    print(f"Processing sliding windows (size={window_size})...")
    for i in tqdm(range(len(df) - window_size + 1)):
        window = features[i : i + window_size]
        target_idx = i + window_size - 1
        target_label = labels[target_idx]
        
        X_windows.append(window)
        y_targets.append(target_label)
        target_indices.append(target_idx)
        
    return np.array(X_windows), np.array(y_targets), np.array(target_indices)

# xây dựng đồ thị và trích xuất embeddings GAT cho tập valid và test
device = 'cuda' if torch.cuda.is_available() else 'cpu'

num_features = len([c for c in data_valid.columns if c not in ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']])
num_classes = len(data_valid['label'].unique())

# load gat embedder
print(f"Đang Load trọng số GAT_Embedder (features = {num_features}, classes = {num_classes})...")
gat_model = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=num_classes, heads=8, edge_dim=4).to(device)
gat_model.load_state_dict(torch.load(r"model_final\gat_embedder_24_4_best.pth", map_location=device))

# xây dựng inference graph (tách riêng cho valid và test để mô phỏng thực tế, không gộp lại giống lúc train)
print("\n--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---")
graph_train, data_train = build_inference_graph(data_train)
graph_valid, data_valid = build_inference_graph(data_valid)
graph_test, data_test = build_inference_graph(data_test)

# trích xuất embeddings từ valid và test
print("\n--- TRÍCH XUẤT EMBEDDINGS TỪ VALID VÀ TEST ---")
train_gat_emb = extract_embeddings_mask(gat_model, graph_train, device=device)
valid_gat_emb = extract_embeddings_mask(gat_model, graph_valid, device=device)
test_gat_emb  = extract_embeddings_mask(gat_model, graph_test, device=device)

# tạo sliding windows cho CNN-BiLSTM (không có under-sampling, giữ nguyên thứ tự thời gian)
X_train_win, y_train_win, train_align_idx = get_window_data_with_index(data_train, window_size=10)
X_val_win, y_val_win, val_align_idx = get_window_data_with_index(data_valid, window_size=10)
X_test_win, y_test_win, test_align_idx = get_window_data_with_index(data_test, window_size=10)

print(f"\n[Valid] Windows Shape: {X_val_win.shape} | Align Indices: {val_align_idx.shape}")
print(f"[Test]  Windows Shape: {X_test_win.shape}  | Align Indices: {test_align_idx.shape}")

# sử dụng mô hình XGBoost đã huấn luyện để trích xuất xác suất dự đoán (probabilities) cho tập train và valid
print("\n--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(r"model_final\GAT_XGB_Hybrid_FAISS_Model.json")

prob_train_xgb = xgb_model.predict_proba(train_gat_emb)
prob_valid_xgb = xgb_model.predict_proba(valid_gat_emb)
prob_test_xgb  = xgb_model.predict_proba(test_gat_emb)

print("\n--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---")
cnn_model = CNN_BiLSTM_Attention(num_features=num_features, num_classes=num_classes, time_steps=10, hidden_size=128).to(device)
cnn_model.load_state_dict(torch.load(r"model_final\best_cnn_bilstm_best_0.9.pth", map_location=device))

# extract probabilities từ CNN-BiLSTM-Attention cho tập valid và test
prob_train_bilstm = extract_probs_cnn_bilstm(cnn_model, X_train_win, batch_size=512, device=device)
prob_valid_bilstm = extract_probs_cnn_bilstm(cnn_model, X_val_win, batch_size=512, device=device)
prob_test_bilstm  = extract_probs_cnn_bilstm(cnn_model, X_test_win, batch_size=512, device=device)


# rút ra các dự đoán XGBoost (Node level) tương ứng với phần tử cuối cùng của (Sequence Level window)
aligned_prob_train_xgb = prob_train_xgb[train_align_idx]
aligned_prob_valid_xgb = prob_valid_xgb[val_align_idx]
aligned_prob_test_xgb  = prob_test_xgb[test_align_idx]

# nối đặc trưng meta: XGB Probabilities + CNN-BiLSTM Probabilities

X_meta_train = np.concatenate([aligned_prob_train_xgb, prob_train_bilstm], axis=1)
y_meta_train = y_train_win

X_meta_valid = np.concatenate([aligned_prob_valid_xgb, prob_valid_bilstm], axis=1)
y_meta_valid = y_val_win 

X_meta_test = np.concatenate([aligned_prob_test_xgb, prob_test_bilstm], axis=1)
y_meta_test = y_test_win

print(f"\n[Meta-Dataset] Train Shape:      {X_meta_train.shape}")
print(f"\n[Meta-Dataset] Validation Shape: {X_meta_valid.shape}")
print(f"[Meta-Dataset] Test Shape:       {X_meta_test.shape}")


# meta-learner dùng XGBoost
print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---")

meta_model = xgb.XGBClassifier(
    n_estimators=300,          
    max_depth=3,           
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    early_stopping_rounds=20  
)

meta_model.fit(
    X_meta_valid, y_meta_valid,
    eval_set=[(X_meta_valid, y_meta_valid)],

    verbose=False
)

final_preds = meta_model.predict(X_meta_test)
print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, final_preds, digits=4))

Đang Load trọng số GAT_Embedder (features = 314, classes = 11)...

--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---
Pre-processing Timestamp cho Inference Graph...


C:\Users\Admin\AppData\Local\Temp\ipykernel_6100\1644161619.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gat_model.load_state_dict(torch.load(r"model_final\gat_embed

[1/2] Build IP Dictionary:   0%|          | 0/2470638 [00:00<?, ?it/s]

[2/2] Nối lưới đồ thị (1-chiều Inference):   0%|          | 0/23 [00:00<?, ?it/s]

Pre-processing Timestamp cho Inference Graph...


...

Processing sliding windows (size=10)...


  0%|          | 0/760231 [00:00<?, ?it/s]


[Valid] Windows Shape: (570125, 10, 314) | Align Indices: (570125,)
[Test]  Windows Shape: (760231, 10, 314)  | Align Indices: (760231,)

--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---

--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_6100\1644161619.py:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(r"model_final\best_cnn_

X_windows.shape: (2470629, 10, 314) dtype: float32 bytes: 31031100240
X_windows.shape: (570125, 10, 314) dtype: float32 bytes: 7160770000
X_windows.shape: (760231, 10, 314) dtype: float32 bytes: 9548501360

[Meta-Dataset] Train Shape:      (2470629, 22)

[Meta-Dataset] Validation Shape: (570125, 22)
[Meta-Dataset] Test Shape:       (760231, 22)

--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---
Accuracy:         99.23%
F1-Score (Macro): 96.97%

Classification Report:

              precision    recall  f1-score   support

           0     0.9677    0.9896    0.9786     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.9559    0.9909    0.9731      2515
           3     0.9997    0.9665    0.9828     36253
           4     0.9439    0.9135    0.9284     18979
           5     1.0000    1.0000    1.0000      2451
           6     0.9408    0.8313    0.8826     11847
           7     1.0

In [10]:
# lưu x_meta_train, y_meta_train và x_meta_valid, y_meta_valid, x_meta_test, y_meta_test ra file để huấn luyện meta-learner sau này (nếu cần)
import joblib
joblib.dump(X_meta_train, "X_meta_train.pkl")
joblib.dump(y_meta_train, "y_meta_train.pkl")
joblib.dump(X_meta_valid, "X_meta_valid.pkl")
joblib.dump(y_meta_valid, "y_meta_valid.pkl")
joblib.dump(X_meta_test, "X_meta_test.pkl")
joblib.dump(y_meta_test, "y_meta_test.pkl")

['y_meta_test.pkl']

Meta Learner bằng Logistic Regression

In [1]:
import joblib

X_meta_train = joblib.load("X_meta_train.pkl")
y_meta_train = joblib.load("y_meta_train.pkl")
X_meta_valid = joblib.load("X_meta_valid.pkl")
y_meta_valid = joblib.load("y_meta_valid.pkl")
X_meta_test = joblib.load("X_meta_test.pkl")
y_meta_test = joblib.load("y_meta_test.pkl")

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER (LOGISTIC REGRESSION) ---")

# 1. Khởi tạo mô hình với các tham số tối ưu nhất cho Stacking Probabilities
best_lr_model = LogisticRegression(
    C=1.0, 
    solver='lbfgs', 
    class_weight='balanced', 
    max_iter=2000, 
    random_state=42
)

# 2. Huấn luyện trực tiếp trên tập Valid (không qua GridSearchCV)
print("Đang huấn luyện mô hình Meta-learner trên tập Valid...")
best_lr_model.fit(X_meta_valid, y_meta_valid)

# 3. Dự đoán và đánh giá trên tập Test
lr_final_preds = best_lr_model.predict(X_meta_test)

print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, lr_final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, lr_final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, lr_final_preds, digits=4))


--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER (LOGISTIC REGRESSION) ---
Đang huấn luyện mô hình Meta-learner trên tập Valid...

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE ---
Accuracy:         98.94%
F1-Score (Macro): 96.38%

Classification Report:

              precision    recall  f1-score   support

           0     0.9716    0.9952    0.9833     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.9992    0.9881    0.9936      2515
           3     0.9997    0.9711    0.9852     36253
           4     0.8856    0.8895    0.8876     18979
           5     1.0000    1.0000    1.0000      2451
           6     0.9014    0.8585    0.8794     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8829    0.9968    0.9364     16746
           9     0.9529    0.9328    0.9427     41514
          10     0.9995    0.9873    0.9934     18567

    accuracy                         0.9894    760231
   macro avg     0.9630    0.9654    0.9638   

In [9]:
# đánh giá chất lượng nhãn trên tập valid (để xem hiệu quả base learner trước khi stacking)

# 11 cột đầu tiên là xác suất từ XGBoost, 11 cột tiếp theo là xác suất từ CNN-BiLSTM-Attention
x_prob_xgb = X_meta_test[:, :11]  # Xác suất từ XGBoost
prob_bilstm = X_meta_test[:, 11:]  # Xác suất từ CNN-BiLSTM-Attention
y_true = y_meta_test

# in ra classification report của từng base learner để đánh giá hiệu quả trước khi stacking
print("\n--- ĐÁNH GIÁ CHẤT LƯỢNG NHÃN TRÊN TẬP VALID (TRƯỚC KHI STACKING) ---")
print("\n[Base Learner 1: XGBoost] ---")
print(classification_report(y_true, np.argmax(x_prob_xgb, axis=1), digits=4))
print("\n[Base Learner 2: CNN-BiLSTM-Attention] ---")
print(classification_report(y_true, np.argmax(prob_bilstm, axis=1), digits=4))


--- ĐÁNH GIÁ CHẤT LƯỢNG NHÃN TRÊN TẬP VALID (TRƯỚC KHI STACKING) ---

[Base Learner 1: XGBoost] ---
              precision    recall  f1-score   support

           0     0.9103    0.9966    0.9515     19846
           1     0.9808    1.0000    0.9903    484077
           2     1.0000    0.9845    0.9922      2515
           3     0.9980    0.9724    0.9850     36253
           4     0.7790    0.8598    0.8174     18979
           5     1.0000    1.0000    1.0000      2451
           6     0.8055    0.7491    0.7763     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8490    0.9972    0.9172     16746
           9     0.9999    0.6666    0.7999     41514
          10     0.9993    0.9654    0.9821     18567

    accuracy                         0.9720    760231
   macro avg     0.9383    0.9265    0.9283    760231
weighted avg     0.9734    0.9720    0.9706    760231


[Base Learner 2: CNN-BiLSTM-Attention] ---
              precision    recall  f1-scor

In [11]:
# lưu tạm x_meta_train, y_meta_train, X_meta_valid, y_meta_valid, X_meta_test, y_meta_test để có thể load lại nhanh khi cần debug hoặc thử nghiệm thêm các meta-learner khác mà không phải chạy lại toàn bộ pipeline từ đầu
import joblib
joblib.dump(X_meta_train, "X_meta_train.pkl")
joblib.dump(y_meta_train, "y_meta_train.pkl")
joblib.dump(X_meta_valid, "X_meta_valid.pkl")
joblib.dump(y_meta_valid, "y_meta_valid.pkl")
joblib.dump(X_meta_test, "X_meta_test.pkl")
joblib.dump(y_meta_test, "y_meta_test.pkl")


['y_meta_test.pkl']

Thêm đặc trưng tương tác (Interaction Term): [p1, p2, p1 * p2] cho Logistic Regression

In [ ]:
print("\n--- BẮT ĐẦU TUNING META-LEARNER VỚI ĐẶC TRƯNG TƯƠNG TÁC (INTERACTION TERMS) ---")
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
# rút ra dự đoán từ XGBoost (là 11 cột đầu tiền của X_train_meta) và CNN-BiLSTM-Attention (11 cột tiếp theo của X_train_meta)
aligned_prob_valid_xgb = X_meta_train[:, :11]  # Xác suất từ XGBoost
prob_valid_bilstm = X_meta_train[:, 11:]       # Xác suất từ CNN-BiLSTM-Attention
y_meta_valid = y_meta_train

# tương tự cho tập test
aligned_prob_test_xgb = X_meta_test[:, :11]
prob_test_bilstm = X_meta_test[:, 11:]
y_meta_test = y_meta_test

# Tạo đặc trưng tương tác: p1 * p2
interaction_valid = aligned_prob_valid_xgb * prob_valid_bilstm
interaction_test  = aligned_prob_test_xgb  * prob_test_bilstm

# Nối các đặc trưng: [p1, p2, p1 * p2]
X_meta_valid_inter = np.concatenate([aligned_prob_valid_xgb, prob_valid_bilstm, interaction_valid], axis=1)
X_meta_test_inter  = np.concatenate([aligned_prob_test_xgb, prob_test_bilstm, interaction_test], axis=1)

print(f"[Meta-Dataset with Interaction] Validation Shape: {X_meta_valid_inter.shape}")
print(f"[Meta-Dataset with Interaction] Test Shape:       {X_meta_test_inter.shape}")
# Sử dụng trực tiếp cấu hình siêu tham số tốt nhất
best_lr_inter_model = LogisticRegression(
    C=0.1, 
    class_weight='balanced', 
    solver='liblinear', 
    max_iter=2000, 
    random_state=42
)

print("Đang huấn luyện Logistic Regression (Interaction) với bộ param tốt nhất...")
best_lr_inter_model.fit(X_meta_valid_inter, y_meta_valid)

# Sử dụng mô hình tốt nhất cho tập test thực tế
lr_inter_final_preds = best_lr_inter_model.predict(X_meta_test_inter)

print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (LOGISTIC REGRESSION + INTERACTION) ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, lr_inter_final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, lr_inter_final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, lr_inter_final_preds, digits=4))

In [ ]:
# train XGBoost meta-learner với đặc trưng tương tác
print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost VỚI ĐẶC TRƯNG TƯƠNG TÁC ---")
meta_model_inter = xgb.XGBClassifier(
    n_estimators=300,      
    max_depth=3,           
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)
meta_model_inter.fit(X_meta_valid_inter, y_meta_valid, eval_set=[(X_meta_test_inter, y_meta_test)], verbose=False)
final_preds_inter = meta_model_inter.predict(X_meta_test_inter)
print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (XGBOOST META-LEARNER + INTERACTION) ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, final_preds_inter)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, final_preds_inter, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, final_preds_inter, digits=4))
